# 0 — Deteksi Manusia & Cropping Identitas (YOLOv8n)

Notebook persiapan data untuk `identity-lora-pipeline`: mendeteksi manusia pada foto
mentah per identitas dengan YOLOv8n (kelas `person`), memvalidasi tepat satu orang per
foto, lalu memotong region sesuai bounding box (dengan padding) menjadi crop siap pakai
untuk tahap alignment berikutnya.

Alur: `data/raw/<identitas>/` → deteksi → validasi tepat-1-box → crop + padding →
`data/cropped/<identitas>/`. Foto dengan 0 atau lebih dari satu box `person` dilewati dan
dicatat ke log, bukan diproses otomatis (cakupan MVP: satu orang per foto).


## 1. Import dan logger


In [ ]:
import logging
import sys
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

# Contoh: 2026-09-26 10:30:00,123 | INFO     | human_detection_and_cropping | Dilewati ...
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"


def setup_logger(name: str) -> logging.Logger:
    """Membuat logger dengan format teks.

    Args:
        name: Nama logger.

    Returns:
        Logger yang siap dipakai.
    """
    created = logging.getLogger(name)
    if created.handlers:
        return created

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(LOG_FORMAT))
    created.addHandler(handler)
    created.setLevel(logging.INFO)
    return created


logger = setup_logger("human_detection_and_cropping")


## 2. Konfigurasi


In [ ]:
RAW_DIR = Path("data/raw")
CROPPED_DIR = Path("data/cropped")

MODEL_WEIGHTS = "yolov8n.pt"
PERSON_CLASS_ID = 0  # id kelas "person" pada dataset COCO yang dipakai YOLOv8n
CONF_THRESHOLD = 0.5  # sementara, belum dikalibrasi pada data asli (K1)

PADDING_RATIO = 0.1  # padding simetris di semua sisi bounding box sebelum crop (K4)
MIN_SIDE_PX = 64  # sisi terpanjang minimum hasil crop, sementara (K3)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


## 3. Error pipeline


In [ ]:
class ImageReadError(Exception):
    """Gambar mentah gagal dibaca."""


class DetectionError(Exception):
    """Model deteksi gagal dijalankan pada gambar."""


class NoDetectionError(Exception):
    """Tidak ada box "person" yang terdeteksi pada gambar."""


class MultipleDetectionError(Exception):
    """Lebih dari satu box "person" terdeteksi pada gambar (di luar cakupan MVP)."""


class CropTooSmallError(Exception):
    """Sisi terpanjang hasil crop di bawah ukuran minimum."""


class CropSaveError(Exception):
    """Hasil crop gagal disimpan ke berkas."""


## 4. Deteksi manusia (K1)


In [ ]:
@dataclass
class BoundingBox:
    """Satu box hasil deteksi kelas "person"."""

    x1: int
    y1: int
    x2: int
    y2: int
    confidence: float


class HumanDetector:
    """Membungkus YOLOv8n dan memfilter deteksi ke kelas "person" saja."""

    def __init__(self, weights: str, conf_threshold: float) -> None:
        self._model = YOLO(weights)
        self._conf_threshold = conf_threshold

    def _to_boxes(self, results) -> list[BoundingBox]:
        boxes: list[BoundingBox] = []
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                boxes.append(
                    BoundingBox(
                        x1=int(x1),
                        y1=int(y1),
                        x2=int(x2),
                        y2=int(y2),
                        confidence=float(box.conf[0]),
                    )
                )
        return boxes

    def detect(self, image_path: Path) -> list[BoundingBox]:
        """Mendeteksi box "person" pada satu gambar.

        Args:
            image_path: Path ke gambar yang akan dideteksi.

        Returns:
            Daftar box "person" beserta confidence-nya.

        Raises:
            DetectionError: Kalau model gagal melakukan inferensi pada gambar.
        """
        try:
            results = self._model.predict(
                source=str(image_path),
                classes=[PERSON_CLASS_ID],
                conf=self._conf_threshold,
                verbose=False,
            )
        except (RuntimeError, OSError, cv2.error) as e:
            logger.error(f"Deteksi gagal pada {image_path}: {e}", exc_info=True)
            raise DetectionError(f"Deteksi gagal pada {image_path}: {e}") from e
        return self._to_boxes(results)


## 5. Validasi jumlah deteksi (K2)


In [ ]:
def validate_single_detection(boxes: list[BoundingBox]) -> BoundingBox:
    """Memastikan tepat satu box "person" pada satu gambar.

    Args:
        boxes: Daftar box hasil `HumanDetector.detect`.

    Returns:
        Satu-satunya box yang valid.

    Raises:
        NoDetectionError: Kalau tidak ada box sama sekali.
        MultipleDetectionError: Kalau ada lebih dari satu box (di luar cakupan MVP).
    """
    if len(boxes) == 0:
        raise NoDetectionError('Tidak ada box "person" terdeteksi')
    if len(boxes) > 1:
        raise MultipleDetectionError(
            f'{len(boxes)} box "person" terdeteksi, di luar cakupan MVP'
        )
    return boxes[0]


## 6. Cropping (K3, K4, K5)


In [ ]:
class Cropper:
    """Memotong region bounding box dengan padding, dan menyimpan hasilnya."""

    def __init__(self, padding_ratio: float, min_side_px: int) -> None:
        self._padding_ratio = padding_ratio
        self._min_side_px = min_side_px

    def _apply_padding(
        self, box: BoundingBox, image_shape: tuple[int, int]
    ) -> tuple[int, int, int, int]:
        height, width = image_shape
        box_width = box.x2 - box.x1
        box_height = box.y2 - box.y1
        pad_x = int(box_width * self._padding_ratio)
        pad_y = int(box_height * self._padding_ratio)

        x1 = max(box.x1 - pad_x, 0)
        y1 = max(box.y1 - pad_y, 0)
        x2 = min(box.x2 + pad_x, width)
        y2 = min(box.y2 + pad_y, height)
        return x1, y1, x2, y2

    def _is_large_enough(self, region: np.ndarray) -> bool:
        height, width = region.shape[:2]
        return max(height, width) >= self._min_side_px

    def crop(self, image: np.ndarray, box: BoundingBox) -> np.ndarray:
        """Memotong region sesuai box dengan padding, menolak hasil yang terlalu kecil.

        Args:
            image: Gambar asal, hasil `cv2.imread`.
            box: Box "person" yang sudah divalidasi tepat satu.

        Returns:
            Region gambar hasil crop.

        Raises:
            CropTooSmallError: Kalau sisi terpanjang hasil crop di bawah `MIN_SIDE_PX`.
        """
        x1, y1, x2, y2 = self._apply_padding(box, image.shape[:2])
        region = image[y1:y2, x1:x2]
        if not self._is_large_enough(region):
            raise CropTooSmallError(
                f"Crop {max(region.shape[:2])}px di bawah ambang {self._min_side_px}px"
            )
        return region

    def save(self, region: np.ndarray, output_path: Path) -> None:
        """Menyimpan hasil crop ke berkas.

        Args:
            region: Region hasil `crop`.
            output_path: Path tujuan penyimpanan.

        Raises:
            CropSaveError: Kalau berkas gagal ditulis.
        """
        try:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            ok = cv2.imwrite(str(output_path), region)
        except OSError as e:
            logger.error(f"Crop gagal disimpan ke {output_path}: {e}", exc_info=True)
            raise CropSaveError(f"Crop gagal disimpan ke {output_path}: {e}") from e
        if not ok:
            raise CropSaveError(f"Crop gagal disimpan ke {output_path}")


def build_crop_filename(source_path: Path) -> str:
    """Membentuk nama file hasil crop dari nama file asal, agar re-run menimpa (K5).

    Args:
        source_path: Path gambar asal.

    Returns:
        Nama file hasil crop, mis. "foto1_person.jpg".
    """
    return f"{source_path.stem}_person.jpg"


## 7. Orkestrasi per identitas


In [ ]:
class ProcessOutcome(Enum):
    """Kategori hasil pemrosesan satu gambar."""

    CROPPED = "cropped"
    SKIPPED_NO_DETECTION = "skipped_no_detection"
    SKIPPED_MULTIPLE_DETECTION = "skipped_multiple_detection"
    SKIPPED_TOO_SMALL = "skipped_too_small"
    SKIPPED_UNREADABLE = "skipped_unreadable"


@dataclass
class ProcessSummary:
    """Ringkasan hasil pemrosesan satu identitas."""

    cropped: int = 0
    skipped_no_detection: int = 0
    skipped_multiple_detection: int = 0
    skipped_too_small: int = 0
    skipped_unreadable: int = 0

    @property
    def total_skipped(self) -> int:
        return (
            self.skipped_no_detection
            + self.skipped_multiple_detection
            + self.skipped_too_small
            + self.skipped_unreadable
        )


def _read_image(image_path: Path) -> np.ndarray:
    try:
        image = cv2.imread(str(image_path))
    except cv2.error as e:
        logger.error(f"Gambar tidak bisa dibaca: {image_path}: {e}", exc_info=True)
        raise ImageReadError(f"Gambar tidak bisa dibaca: {image_path}") from e
    if image is None:
        raise ImageReadError(f"Gambar tidak bisa dibaca (format tidak dikenali): {image_path}")
    return image


def _list_images(identity_dir: Path) -> list[Path]:
    return sorted(p for p in identity_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def _process_image(
    image_path: Path,
    output_dir: Path,
    detector: HumanDetector,
    cropper: Cropper,
) -> ProcessOutcome:
    try:
        image = _read_image(image_path)
    except ImageReadError as e:
        logger.warning(str(e))
        return ProcessOutcome.SKIPPED_UNREADABLE

    boxes = detector.detect(image_path)
    try:
        box = validate_single_detection(boxes)
    except NoDetectionError:
        logger.info(f'Dilewati, tidak ada deteksi "person": {image_path}')
        return ProcessOutcome.SKIPPED_NO_DETECTION
    except MultipleDetectionError as e:
        logger.warning(f"Dilewati, {e}: {image_path}")
        return ProcessOutcome.SKIPPED_MULTIPLE_DETECTION

    try:
        region = cropper.crop(image, box)
    except CropTooSmallError as e:
        logger.warning(f"Dilewati, {e}: {image_path}")
        return ProcessOutcome.SKIPPED_TOO_SMALL

    output_path = output_dir / build_crop_filename(image_path)
    cropper.save(region, output_path)
    return ProcessOutcome.CROPPED


def process_identity(
    identity_dir: Path,
    output_dir: Path,
    detector: HumanDetector,
    cropper: Cropper,
) -> ProcessSummary:
    """Menjalankan deteksi, validasi, dan cropping untuk satu identitas.

    Args:
        identity_dir: Folder foto mentah identitas, dari `data/raw/<identitas>/`.
        output_dir: Folder tujuan hasil crop, `data/cropped/<identitas>/`.
        detector: `HumanDetector` yang dipakai bersama untuk semua identitas.
        cropper: `Cropper` yang dipakai bersama untuk semua identitas.

    Returns:
        Ringkasan jumlah crop dan gambar yang dilewati per kategori.
    """
    if not identity_dir.exists():
        logger.warning(f"Folder identitas tidak ditemukan: {identity_dir}")
        return ProcessSummary()

    output_dir.mkdir(parents=True, exist_ok=True)
    summary = ProcessSummary()
    for image_path in _list_images(identity_dir):
        outcome = _process_image(image_path, output_dir, detector, cropper)
        setattr(summary, outcome.value, getattr(summary, outcome.value) + 1)

    logger.info(
        f"[{identity_dir.name}] {summary.cropped} crop tersimpan di {output_dir}, "
        f"{summary.total_skipped} dilewati"
    )
    return summary


def run_pipeline(identities: dict[str, Path]) -> dict[str, ProcessSummary]:
    """Menjalankan pipeline untuk setiap identitas.

    Args:
        identities: Pemetaan nama identitas ke folder `data/raw/<identitas>/`.

    Returns:
        Ringkasan hasil pemrosesan per identitas.
    """
    detector = HumanDetector(weights=MODEL_WEIGHTS, conf_threshold=CONF_THRESHOLD)
    cropper = Cropper(padding_ratio=PADDING_RATIO, min_side_px=MIN_SIDE_PX)

    summaries: dict[str, ProcessSummary] = {}
    for identity_name, identity_dir in identities.items():
        output_dir = CROPPED_DIR / identity_name
        summaries[identity_name] = process_identity(identity_dir, output_dir, detector, cropper)
    return summaries


## 8. Jalankan pipeline untuk identitas nyata

Isi `data/raw/<identitas>/` dengan foto identitas sebelum menjalankan sel ini. Sel ini
mengunduh bobot `yolov8n.pt` (butuh koneksi internet sekali) dan memanggil model
sungguhan — sengaja tidak dijalankan otomatis saat pemeriksaan kode (lihat README.md).


In [ ]:
IDENTITIES = {
    identity_dir.name: identity_dir
    for identity_dir in sorted(RAW_DIR.iterdir())
    if identity_dir.is_dir()
} if RAW_DIR.exists() else {}

summaries = run_pipeline(IDENTITIES)
for identity_name, summary in summaries.items():
    print(f"{identity_name}: {summary.cropped} crop, {summary.total_skipped} dilewati")
